# COMP365 Team Project: AI-Based Technical Language Simplifier

**Team Name:** MindStack

**Team Members:**
- Derry Moss
- Jordyn Walsh-Hill
- Sydney Jackson
- Jerry Kang

## Project Overview

This project explores how a large language model (LLM) can simplify technical documentation into language that is easier for middle-school readers to understand while preserving the original technical meaning.

## Dataset

The dataset contains 30 technical passages collected from technical documentation sources. Each passage includes a passage ID, category, source URL, original text, and word count.

## Prototype Objectives

1. Load and inspect the technical passage dataset.
2. Use an existing language model to simplify technical text.
3. Compare original and simplified passages.
4. Measure readability changes.
5. Evaluate whether the simplified text preserves the original meaning and remains factually accurate.

## Current Stage

Practical Checkpoint 1: Initial prototype development and evaluation.


## 1. Connect Google Drive

Google Drive must be connected to Google Colab so the notebook can access the project dataset, `passage.csv`.

The connection allows the notebook to read files stored in the `COMP365_Team_Project` folder.

**Expected result:** Google Drive is mounted successfully, and Colab can access the project files.


In [63]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



## 2. Locate the Dataset

The project dataset, `passage.csv`, is stored in the `COMP365_Team_Project` folder in Google Drive.

This step checks whether the notebook can access the project folder and locate the dataset before loading it.

**Expected result:** The files stored in the project folder are displayed.


In [64]:

import os

folder_path = "/content/drive/MyDrive/COMP365_Team_Project"

print(os.listdir(folder_path))


['COMP365_Technical_Language_Simplifier_Prototype.ipynb', 'Passages.csv', 'Simplified_Passages.csv', 'Quality_Review.csv', 'Quality_Review_Summary.csv', 'Readability_Results.csv']


In [65]:

import os

# Search for the shared project folder in Google Drive
for root, dirs, files in os.walk("/content/drive"):
    if "Passages.csv" in files:
        print("FOUND Passages.csv:", os.path.join(root, "Passages.csv"))


FOUND Passages.csv: /content/drive/.shortcut-targets-by-id/1LOPJ1Y8-Xp0iCAEax_0CCgO66y633kem/COMP365_Team_Project/Passages.csv
FOUND Passages.csv: /content/drive/MyDrive/Passages.csv
FOUND Passages.csv: /content/drive/MyDrive/COMP365_Team_Project/Passages.csv



## 3. Load the Dataset

The `Passages.csv` file contains the 30 technical passages collected for this project.

This step loads the CSV file into a pandas DataFrame, a table-like structure that allows the dataset to be inspected and analyzed using Python.

**Expected result:** The dataset loads successfully, and the first five rows are displayed.


In [66]:

import pandas as pd
import os

project_folder = "/content/drive/.shortcut-targets-by-id/1LOPJ1Y8-Xp0iCAEax_0CCgO66y633kem/COMP365_Team_Project"

file_path = os.path.join(project_folder, "Passages.csv")

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of passages:", len(df))
display(df.head())



Dataset loaded successfully!
Number of passages: 30


,Passage_ID,Category,Source_URL,Original_Text,Word_Count
0,P001,Cloud Computing,https://learn.microsoft.com/en-us/azure/cloud-...,Microsoft Azure is Microsoft's cloud platform ...,46
1,P002,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,When you send a request through any of the Azu...,53
2,P003,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,All capabilities that are available in the Azu...,38
3,P004,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource - A manageable item that's available ...,33
4,P005,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource group - A container that holds relate...,46


In [67]:

simplified_path = os.path.join(project_folder, "Simplified_Passages.csv")

simplified_df = pd.read_csv(simplified_path)

print("Simplified passages loaded:", len(simplified_df))
display(simplified_df.head())


Simplified passages loaded: 30


,Passage_ID,Category,Source_URL,Original_Text,Word_Count,Simplified_Text
0,P001,Cloud Computing,https://learn.microsoft.com/en-us/azure/cloud-...,Microsoft Azure is Microsoft's cloud platform ...,46,Microsoft Azure is like a big playground where...
1,P002,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,When you send a request through any of the Azu...,53,"When you use any Azure tool, like an API or SD..."
2,P003,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,All capabilities that are available in the Azu...,38,All things you can do in the Azure portal work...
3,P004,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource - A manageable item that's available ...,33,A resource is like a tool you can use in Azure...
4,P005,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource group - A container that holds relate...,46,A resource group is like a special box where y...



## 4. Inspect the Dataset

The dataset contains technical passages from several categories. Before processing the text, it is important to check the dataset's dimensions, column names, and category distribution.

This step summarizes the dataset and counts how many passages belong to each category.

**Expected result:** The dataset dimensions, column names, and number of passages in each category are displayed.


In [68]:

print("Dataset dimensions (rows, columns):")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nPassages by category:")
print(df["Category"].value_counts())


Dataset dimensions (rows, columns):
(30, 5)

Column names:
['Passage_ID', 'Category', 'Source_URL', 'Original_Text', 'Word_Count']

Passages by category:
Category
Cloud Computing         10
Databases                5
Networking               5
Security                 5
Software Engineering     5
Name: count, dtype: int64



## 5. Check Data Quality

Before processing the technical passages, the dataset must be checked for missing values, duplicate passage IDs, and empty text entries.

These checks help identify data problems that could affect the prototype's results.

**Expected result:** The number of missing values, duplicate passage IDs, and empty original text entries is displayed.


In [69]:

print("Missing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate passage IDs:")
print(df["Passage_ID"].duplicated().sum())

print("\nNumber of empty original text entries:")
print(df["Original_Text"].fillna("").str.strip().eq("").sum())


Missing values in each column:
Passage_ID       0
Category         0
Source_URL       0
Original_Text    0
Word_Count       0
dtype: int64

Number of duplicate passage IDs:
0

Number of empty original text entries:
0


## 6. Select and Load the Language Model

This project uses Qwen2.5-1.5B-Instruct, a language model that can rewrite technical passages in simpler language.
The goal is to make technical writing easier for middle-school students while preserving its essential meaning. The model will run in Google Colab without requiring a paid API.

In [70]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully:", model_name)
print("Device:", model.device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully: Qwen/Qwen2.5-1.5B-Instruct
Device: cpu


## 7. Define the Simplification Evaluation Criteria

The purpose of this project is to make technical passages easier for middle-school students to understand while preserving their essential meaning.

Each simplified passage will be evaluated using three criteria:

- **Readability:** Is the rewritten passage easier to understand?
- **Meaning preservation:** Does it communicate the same essential ideas as the original?
- **Factual accuracy:** Does it avoid misleading statements and unsupported claims?

The evaluation will allow different wording and sentence structures. A successful simplification does not need to match the original word for word, but it should preserve the important information.

These criteria will guide the model experiments and the final evaluation of the 30 passages.

In [28]:

# Define the evaluation criteria for the project

evaluation_criteria = {
    "Readability": "Is the rewritten passage easier to understand?",
    "Meaning Preservation": "Does it communicate the same essential ideas?",
    "Factual Accuracy": "Does it avoid misleading or unsupported claims?"
}

for criterion, description in evaluation_criteria.items():
    print(f"{criterion}: {description}")


Readability: Is the rewritten passage easier to understand?
Meaning Preservation: Does it communicate the same essential ideas?
Factual Accuracy: Does it avoid misleading or unsupported claims?


## 8. Prepare the Batch Simplification

The dataset contains 30 technical passages from five categories. This project will use an automated workflow to simplify the passages and measure readability changes.

The language model will rewrite technical passages using simpler words and shorter sentences while preserving their essential meaning. Readability scores will be calculated automatically for the original and simplified text.

Meaning preservation and factual accuracy will be reviewed separately because readability scores alone cannot confirm that a rewrite is correct. A small batch will be tested before processing the complete dataset.

In [29]:

# Confirm the dataset is ready for batch processing

print("Total passages:", len(df))
print("\nPassages by category:")
print(df["Category"].value_counts())

print("\nRequired columns present:")
required_columns = ["Passage_ID", "Category", "Original_Text"]
print(all(column in df.columns for column in required_columns))

print("\nMissing original passages:", df["Original_Text"].isna().sum())


Total passages: 30

Passages by category:
Category
Cloud Computing         10
Databases                5
Networking               5
Security                 5
Software Engineering     5
Name: count, dtype: int64

Required columns present:
True

Missing original passages: 0


## 9. Simplify a Small Batch of Passages

The first three technical passages will be used to test the language model's ability to simplify technical content.

The model will be instructed to use common words and clear sentences, preserve the original meaning, and avoid adding facts, examples, or comparisons that are not present in the source.

The generated passages will be reviewed before the workflow is applied to the full dataset. This initial test helps identify whether the model is suitable for the project.

In [30]:
# Simplify the first three passages as a test batch

import torch

batch_df = df.head(3).copy()

def simplify_text(text):
    prompt = f"""Rewrite the technical passage for a middle-school reader.

Rules:
- Use simple, direct words and clear sentences.
- Preserve the original meaning and important technical facts.
- Do not add facts, examples, metaphors, or comparisons.
- Do not claim that something is faster, safer, or more reliable unless the source says so.
- Keep important technical names and terms when needed.
- Return only the rewritten passage.

Original passage:
{text}"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    simplified_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return simplified_text.strip()


batch_df["Simplified_Text"] = ""

for index, row in batch_df.iterrows():
    print(f"\n{'=' * 70}")
    print(f"Passage: {row['Passage_ID']} | Category: {row['Category']}")
    print("\nORIGINAL:")
    print(row["Original_Text"])

    simplified = simplify_text(row["Original_Text"])
    batch_df.at[index, "Simplified_Text"] = simplified

    print("\nSIMPLIFIED:")
    print(simplified)

print("\nSmall-batch test completed.")


Passage: P001 | Category: Cloud Computing

ORIGINAL:
Microsoft Azure is Microsoft's cloud platform designed to simplify the process of building, deploying, and managing applications and services through a global network of Microsoft-managed data centers. It offers a wide range of services that allow businesses and developers to create scalable, reliable, and maintainable applications.

SIMPLIFIED:
Microsoft Azure is like a big playground where they put all their computer stuff together in special places called data centers. This helps people make fun things on computers without having to worry too much about how it works. They have lots of different toys (services) you can play with to build cool apps and games. These toys work well together and don't break easily, which makes them great for grown-ups who want to share their creations online.

Passage: P002 | Category: Cloud Computing

ORIGINAL:
When you send a request through any of the Azure APIs, tools, or SDKs, Resource Manager re

## 10. Simplify All 30 Passages

The language model will simplify all 30 technical passages automatically using the same model and instructions. The simplified passages will be stored in a results table alongside the original passage IDs and categories.

**Note:** Processing all 30 passages may take approximately 45 to 50 minutes when running on a CPU. Processing time may vary depending on the available computing resources. Allow the cell to finish before running the next section.

The results will be used for readability measurements, meaning preservation, and factual accuracy evaluation.

In [31]:
# Load the existing simplified passages from Google Drive

import pandas as pd

input_path = (
    "/content/drive/.shortcut-targets-by-id/"
    "1LOPJ1Y8-Xp0iCAEax_0CCgO66y633kem/"
    "COMP365_Team_Project/Simplified_Passages.csv"
)

results_df = pd.read_csv(input_path)

print("Existing simplification results loaded successfully!")
print("Total passages:", len(results_df))

print(
    "Passages with simplified text:",
    results_df["Simplified_Text"].fillna("").str.strip().ne("").sum()
)

display(
    results_df[
        ["Passage_ID", "Category", "Simplified_Text"]
    ].head(5)
)

Existing simplification results loaded successfully!
Total passages: 30
Passages with simplified text: 30


,Passage_ID,Category,Simplified_Text
0,P001,Cloud Computing,Microsoft Azure is like a big playground where...
1,P002,Cloud Computing,"When you use any Azure tool, like an API or SD..."
2,P003,Cloud Computing,All things you can do in the Azure portal work...
3,P004,Cloud Computing,A resource is like a tool you can use in Azure...
4,P005,Cloud Computing,A resource group is like a special box where y...


## 11. Save the Results

The original passages and their simplified versions will be saved to a CSV file in the project folder on Google Drive. This file preserves the passage IDs, categories, source URLs, original text, word counts, and simplified text.

Saving the results makes it possible to continue the evaluation without generating the simplified passages again.

In [32]:
# Save the original and simplified passages to Google Drive

import os

output_path = (
    "/content/drive/MyDrive/COMP365_Team_Project/"
    "Simplified_Passages.csv"
)

results_df.to_csv(output_path, index=False)

print("Results saved successfully!")
print("File location:", output_path)
print("Total passages saved:", len(results_df))
print("Simplified text available:",
      results_df["Simplified_Text"].fillna("").str.strip().ne("").sum(),
      "passages")

Results saved successfully!
File location: /content/drive/MyDrive/COMP365_Team_Project/Simplified_Passages.csv
Total passages saved: 30
Simplified text available: 30 passages


## 12. Measure Readability

Readability scores will be calculated for the original and simplified passages to evaluate whether the language model made the technical content easier to read.

The Flesch Reading Ease score estimates how easy a passage is to read. Higher scores generally indicate easier text. The Flesch-Kincaid Grade Level estimates the U.S. school grade level needed to understand the text.

The scores for the original and simplified passages will be compared to identify changes in readability. These scores measure reading difficulty, not whether the simplified text preserves the original meaning or factual accuracy.

In [33]:
# Install the readability library
!pip -q install textstat


In [34]:
# Calculate readability scores for original and simplified passages

import textstat
import pandas as pd

readability_df = results_df.copy()

readability_df["Original_Flesch_Reading_Ease"] = (
    readability_df["Original_Text"].apply(
        textstat.flesch_reading_ease
    )
)

readability_df["Simplified_Flesch_Reading_Ease"] = (
    readability_df["Simplified_Text"].apply(
        textstat.flesch_reading_ease
    )
)

readability_df["Original_Grade_Level"] = (
    readability_df["Original_Text"].apply(
        textstat.flesch_kincaid_grade
    )
)

readability_df["Simplified_Grade_Level"] = (
    readability_df["Simplified_Text"].apply(
        textstat.flesch_kincaid_grade
    )
)

readability_df["Reading_Ease_Change"] = (
    readability_df["Simplified_Flesch_Reading_Ease"]
    - readability_df["Original_Flesch_Reading_Ease"]
)

readability_df["Grade_Level_Change"] = (
    readability_df["Simplified_Grade_Level"]
    - readability_df["Original_Grade_Level"]
)

print("Readability scores calculated for all passages.")
print("Total passages:", len(readability_df))

display(
    readability_df[
        [
            "Passage_ID",
            "Original_Flesch_Reading_Ease",
            "Simplified_Flesch_Reading_Ease",
            "Original_Grade_Level",
            "Simplified_Grade_Level",
        ]
    ].round(2)
)

Readability scores calculated for all passages.
Total passages: 30


,Passage_ID,Original_Flesch_Reading_Ease,Simplified_Flesch_Reading_Ease,Original_Grade_Level,Simplified_Grade_Level
0,P001,14.29,68.31,16.98,8.78
1,P002,46.84,75.98,11.12,6.08
2,P003,36.16,78.46,12.94,7.04
3,P004,24.16,26.95,12.54,18.44
4,P005,61.89,80.79,8.35,6.96
5,P006,21.47,78.33,12.00,5.32
6,P007,57.31,81.13,9.32,5.83
7,P008,41.63,79.09,11.80,5.54
8,P009,30.20,65.73,12.40,7.57
9,P010,38.06,71.14,10.52,7.50


### 12.1. Investigate an Unusual Readability Score

Passage P023 produced an unusually low Reading Ease score and an extremely high estimated grade level. The original and simplified texts will be inspected to determine whether the result is caused by unusual generated text or a limitation of the readability formula.

In [35]:
# Inspect P023 and its readability measurements

p023 = readability_df.loc[
    readability_df["Passage_ID"] == "P023"
].iloc[0]

print("ORIGINAL TEXT:")
print(p023["Original_Text"])

print("\nSIMPLIFIED TEXT:")
print(p023["Simplified_Text"])

print("\nTEXT LENGTH")
print("Original word count:",
      len(str(p023["Original_Text"]).split()))
print("Simplified word count:",
      len(str(p023["Simplified_Text"]).split()))

print("\nREADABILITY SCORES")
print("Original Reading Ease:",
      p023["Original_Flesch_Reading_Ease"])
print("Simplified Reading Ease:",
      p023["Simplified_Flesch_Reading_Ease"])
print("Original Grade Level:",
      p023["Original_Grade_Level"])
print("Simplified Grade Level:",
      p023["Simplified_Grade_Level"])

ORIGINAL TEXT:
The .NET cryptography system implements an extensible pattern of derived class inheritance. The hierarchy is as follows: Algorithm type class, such as SymmetricAlgorithm, AsymmetricAlgorithm, or HashAlgorithm. This level is abstract. Algorithm class that inherits from an algorithm type class, for example, Aes, RSA, or ECDiffieHellman. This level is abstract. Implementation of an algorithm class that inherits from an algorithm class, for example, AesManaged, RC2CryptoServiceProvider, or ECDiffieHellmanCng. This level is fully implemented. This pattern of derived classes lets you add a new algorithm or a new implementation of an existing algorithm. For example, to create a new public-key algorithm, you would inherit from the AsymmetricAlgorithm class. To create a new implementation of a specific algorithm, you would create a non-abstract derived class of that algorithm.

SIMPLIFIED TEXT:
In the .NET cryptography system, there's a way to make your own special kind of math w

### 12.2. Correct the Language of P023

Passage P023 was initially generated in Chinese instead of English. The issue was corrected, and the final English version is already included in the saved Simplified_Passages.csv file. No additional generation is required.

In [36]:
# P023 correction is already included in Simplified_Passages.csv

print(
    "P023 correction is already included in the saved "
    "Simplified_Passages.csv file. No regeneration needed."
)

P023 correction is already included in the saved Simplified_Passages.csv file. No regeneration needed.


### 12.3. Finalize P023 and Update Readability Scores

The final simplified text for P023 is already saved in the results file. Readability scores are calculated for all 30 passages using the saved original and simplified texts. This step does not modify any passage text; it only produces the readability measurements for analysis.

In [37]:
!pip install textstat

In [38]:
# Recalculate readability scores for all 30 passages

import textstat

readability_df = results_df.copy()

readability_df["Original_Flesch_Reading_Ease"] = (
    readability_df["Original_Text"].apply(textstat.flesch_reading_ease)
)

readability_df["Simplified_Flesch_Reading_Ease"] = (
    readability_df["Simplified_Text"].apply(textstat.flesch_reading_ease)
)

readability_df["Original_Grade_Level"] = (
    readability_df["Original_Text"].apply(textstat.flesch_kincaid_grade)
)

readability_df["Simplified_Grade_Level"] = (
    readability_df["Simplified_Text"].apply(textstat.flesch_kincaid_grade)
)

readability_df["Reading_Ease_Change"] = (
    readability_df["Simplified_Flesch_Reading_Ease"]
    - readability_df["Original_Flesch_Reading_Ease"]
)

readability_df["Grade_Level_Change"] = (
    readability_df["Simplified_Grade_Level"]
    - readability_df["Original_Grade_Level"]
)

print("Readability scores calculated for", len(readability_df), "passages.")

print("\nP023 readability scores:")

display(
    readability_df.loc[
        readability_df["Passage_ID"] == "P023",
        [
            "Passage_ID",
            "Original_Flesch_Reading_Ease",
            "Simplified_Flesch_Reading_Ease",
            "Original_Grade_Level",
            "Simplified_Grade_Level"
        ]
    ].round(2)
)

Readability scores calculated for 30 passages.

P023 readability scores:


,Passage_ID,Original_Flesch_Reading_Ease,Simplified_Flesch_Reading_Ease,Original_Grade_Level,Simplified_Grade_Level
22,P023,25.05,54.99,12.85,9.35


## 13. Summarize Readability Results

The readability scores for the 30 original and simplified passages will be summarized to measure the overall change in reading difficulty.

The summary will compare average Flesch Reading Ease scores and average Flesch-Kincaid Grade Levels before and after simplification. It will also count how many passages improved on each measure.

These results provide an initial quantitative evaluation of readability. They do not independently confirm meaning preservation or factual accuracy.

In [39]:
# Summarize readability changes across all 30 passages

summary = pd.DataFrame({
    "Measure": [
        "Average Flesch Reading Ease",
        "Average Flesch-Kincaid Grade Level"
    ],
    "Original": [
        readability_df["Original_Flesch_Reading_Ease"].mean(),
        readability_df["Original_Grade_Level"].mean()
    ],
    "Simplified": [
        readability_df["Simplified_Flesch_Reading_Ease"].mean(),
        readability_df["Simplified_Grade_Level"].mean()
    ]
})

summary["Change"] = summary["Simplified"] - summary["Original"]

reading_ease_improved = (
    readability_df["Reading_Ease_Change"] > 0
).sum()

grade_level_improved = (
    readability_df["Grade_Level_Change"] < 0
).sum()

print("OVERALL READABILITY SUMMARY")
display(summary.round(2))

print(f"Passages with higher Reading Ease: {reading_ease_improved}/30")
print(f"Passages with lower estimated Grade Level: {grade_level_improved}/30")

print("\nPASSAGE-LEVEL RESULTS")
display(
    readability_df[
        [
            "Passage_ID",
            "Reading_Ease_Change",
            "Grade_Level_Change"
        ]
    ].round(2)
)

OVERALL READABILITY SUMMARY


,Measure,Original,Simplified,Change
0,Average Flesch Reading Ease,36.49,67.32,30.83
1,Average Flesch-Kincaid Grade Level,12.19,7.94,-4.25


Passages with higher Reading Ease: 28/30
Passages with lower estimated Grade Level: 29/30

PASSAGE-LEVEL RESULTS


,Passage_ID,Reading_Ease_Change,Grade_Level_Change
0,P001,54.02,-8.20
1,P002,29.14,-5.04
2,P003,42.30,-5.90
3,P004,2.78,5.91
4,P005,18.90,-1.39
5,P006,56.86,-6.69
6,P007,23.82,-3.49
7,P008,37.46,-6.26
8,P009,35.53,-4.83
9,P010,33.08,-3.02


## 14. Review Meaning Preservation and Factual Accuracy

Readability scores cannot determine whether a simplified passage preserves the original technical meaning. The simplified passages will therefore be reviewed for meaning preservation and factual accuracy.

The review will identify important information that was omitted, technical details that were changed, and unsupported information introduced by the language model. Findings will be recorded to help explain the strengths and limitations of automated simplification.

In [40]:
# Load the existing quality review results

review_path = (
    "/content/drive/MyDrive/COMP365_Team_Project/"
    "Quality_Review.csv"
)

review_df = pd.read_csv(review_path)

print("Existing quality review results loaded successfully!")
print("Total passages:", len(review_df))
print(
    "Completed reviews:",
    review_df["Meaning_Preserved"].notna().sum()
)

display(review_df.head(3))

Existing quality review results loaded successfully!
Total passages: 30
Completed reviews: 30


,Passage_ID,Category,Original_Text,Simplified_Text,Meaning_Preserved,Factually_Accurate,Review_Notes
0,P001,Cloud Computing,Microsoft Azure is Microsoft's cloud platform ...,Microsoft Azure is like a big playground where...,No,No,"Adds unsupported claims about games, children,..."
1,P002,Cloud Computing,When you send a request through any of the Azu...,"When you use any Azure tool, like an API or SD...",Yes,Yes,Preserves the main request-handling process bu...
2,P003,Cloud Computing,All capabilities that are available in the Azu...,All things you can do in the Azure portal work...,Yes,Yes,Preserves the main point and 180-day timeframe...


## 15. Evaluate Simplification Quality

The quality review worksheet will be used to evaluate the 30 simplified passages. Each passage will be checked for whether its original meaning is preserved and whether its technical information is factually accurate.

The review will record any important omissions, changes to technical details, or unsupported information. These findings will complement the readability measurements and provide a more complete evaluation of the language model's performance.

In [41]:
import pandas as pd

review_path = (
    "/content/drive/MyDrive/COMP365_Team_Project/"
    "Quality_Review.csv"
)

review_df = pd.read_csv(review_path).fillna("")

print("Quality review worksheet loaded!")
print("Total passages:", len(review_df))
print("\nColumns:")
print(review_df.columns.tolist())

display(
    review_df[
        [
            "Passage_ID",
            "Category",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ].head(10)
)

Quality review worksheet loaded!
Total passages: 30

Columns:
['Passage_ID', 'Category', 'Original_Text', 'Simplified_Text', 'Meaning_Preserved', 'Factually_Accurate', 'Review_Notes']


,Passage_ID,Category,Meaning_Preserved,Factually_Accurate,Review_Notes
0,P001,Cloud Computing,No,No,"Adds unsupported claims about games, children,..."
1,P002,Cloud Computing,Yes,Yes,Preserves the main request-handling process bu...
2,P003,Cloud Computing,Yes,Yes,Preserves the main point and 180-day timeframe...
3,P004,Cloud Computing,Yes,No,"Most examples are preserved, but describing a ..."
4,P005,Cloud Computing,Yes,Yes,Preserves the purpose of resource groups and t...
5,P006,Cloud Computing,Yes,Yes,Identifies both resource providers and their p...
6,P007,Cloud Computing,Yes,Yes,Preserves the main idea of declarative syntax ...
7,P008,Cloud Computing,No,No,Omits that an ARM template is a JSON file and ...
8,P009,Cloud Computing,Yes,No,Explains the general purpose of a Bicep file b...
9,P010,Cloud Computing,No,No,Does not clearly explain that an extension res...


## 16. Review the First 10 Passages

The first 10 simplified passages will be reviewed for meaning preservation and factual accuracy. The original and simplified texts will be compared to identify important omissions, changes in technical meaning, or unsupported information.

Each passage will receive a review result. The findings will be saved in the quality review worksheet for use in the final project evaluation.

In [42]:
for _, row in review_df.head(10).iterrows():
    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])

    print("\n")

Passage ID: P001
Category: Cloud Computing

ORIGINAL TEXT:
Microsoft Azure is Microsoft's cloud platform designed to simplify the process of building, deploying, and managing applications and services through a global network of Microsoft-managed data centers. It offers a wide range of services that allow businesses and developers to create scalable, reliable, and maintainable applications.

SIMPLIFIED TEXT:
Microsoft Azure is like a big playground where they put all their computer stuff together in special places called data centers. This helps people make fun things on computers without having to worry too much about how it works. They have lots of different games (services) you can play with, which lets grown-ups build cool apps and kids learn new skills.


Passage ID: P002
Category: Cloud Computing

ORIGINAL TEXT:
When you send a request through any of the Azure APIs, tools, or SDKs, Resource Manager receives the request. It authenticates and authorizes the request before forwardin

In [43]:
# Record quality review results for P001-P005

review_results = {
    "P001": (
        "No",
        "No",
        "Adds unsupported claims about games, children, and learning skills. "
        "Omits building, deploying, and managing applications."
    ),
    "P002": (
        "Yes",
        "Yes",
        "Preserves the main request-handling process but omits explicit "
        "authentication and authorization terminology."
    ),
    "P003": (
        "Yes",
        "Yes",
        "Preserves the main point and 180-day timeframe, though "
        "'about 180 days' is less precise."
    ),
    "P004": (
        "Yes",
        "No",
        "Most examples are preserved, but describing a subscription "
        "as 'your account with Azure' is potentially misleading."
    ),
    "P005": (
        "Yes",
        "Yes",
        "Preserves the purpose of resource groups and the choice of "
        "which resources to include. The analogy is informal."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
display(
    review_df.loc[
        review_df["Passage_ID"].isin(review_results.keys()),
        ["Passage_ID", "Meaning_Preserved",
         "Factually_Accurate", "Review_Notes"]
    ]
)

Review results saved!


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
0,P001,No,No,"Adds unsupported claims about games, children,..."
1,P002,Yes,Yes,Preserves the main request-handling process bu...
2,P003,Yes,Yes,Preserves the main point and 180-day timeframe...
3,P004,Yes,No,"Most examples are preserved, but describing a ..."
4,P005,Yes,Yes,Preserves the purpose of resource groups and t...


## 17. Review Passages P006–P010

Passages P006–P010 will be reviewed by comparing the original technical text with the simplified version. Each passage will be evaluated for meaning preservation and factual accuracy.

The review will identify important omissions, changes in technical details, and unsupported information. The results will be added to the existing quality review worksheet.

In [44]:
for _, row in review_df[
    review_df["Passage_ID"].isin(
        ["P006", "P007", "P008", "P009", "P010"]
    )
].iterrows():

    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])
    print()

Passage ID: P006
Category: Cloud Computing

ORIGINAL TEXT:
resource provider - A service that supplies Azure resources. For example, a common resource provider is Microsoft.Compute, which supplies the virtual machine resource. Microsoft.Storage is another common resource provider.

SIMPLIFIED TEXT:
A resource provider is like a helper in Azure who gives you things to use. Think of it as getting toys from your toy box. One type of helper is called Microsoft.Compute. They give you virtual machines, which are like pretend computers you can play with on the internet. Another helper is Microsoft.Storage, who gives you storage space, kind of like a big box where you keep all your favorite toys safe.

Passage ID: P007
Category: Cloud Computing

ORIGINAL TEXT:
declarative syntax - Syntax that lets you state, "Here's what I intend to create," without having to write the sequence of programming commands to create it. ARM templates and Bicep files are examples of declarative syntax. In those file

In [45]:
# Record quality review results for P006-P010

review_results = {
    "P006": (
        "Yes",
        "Yes",
        "Identifies both resource providers and their purposes. "
        "The toy analogy is informal but broadly understandable."
    ),
    "P007": (
        "Yes",
        "Yes",
        "Preserves the main idea of declarative syntax and correctly "
        "mentions ARM templates and Bicep files."
    ),
    "P008": (
        "No",
        "No",
        "Omits that an ARM template is a JSON file and leaves out "
        "deployment scopes. The recipe analogy is too vague."
    ),
    "P009": (
        "Yes",
        "No",
        "Explains the general purpose of a Bicep file but adds an "
        "unsupported comparison that it makes Azure much simpler than before."
    ),
    "P010": (
        "No",
        "No",
        "Does not clearly explain that an extension resource adds "
        "capabilities to another resource. The toy-key analogy misrepresents "
        "role assignments and access permissions."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
print("Passages reviewed:", len(review_results))
display(
    review_df[
        review_df["Passage_ID"].isin(review_results.keys())
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

Review results saved!
Passages reviewed: 5


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
5,P006,Yes,Yes,Identifies both resource providers and their p...
6,P007,Yes,Yes,Preserves the main idea of declarative syntax ...
7,P008,No,No,Omits that an ARM template is a JSON file and ...
8,P009,Yes,No,Explains the general purpose of a Bicep file b...
9,P010,No,No,Does not clearly explain that an extension res...


## 18. Review Passages P011–P015

Passages P011–P015 will be reviewed by comparing the original technical text with the simplified version. Each passage will be evaluated for meaning preservation and factual accuracy.

The review will identify important omissions, changes in technical details, and unsupported information. The findings will be saved in the existing quality review worksheet.

In [46]:
for _, row in review_df[
    review_df["Passage_ID"].isin(
        ["P011", "P012", "P013", "P014", "P015"]
    )
].iterrows():

    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])
    print()

Passage ID: P011
Category: Databases

ORIGINAL TEXT:
A database is a tool for collecting and organizing information. Databases can store information about people, products, orders, or anything else. Many databases start as a list in a word-processing program or spreadsheet. As the list grows bigger, redundancies and inconsistencies begin to appear in the data. The data becomes hard to understand in list form, and there are limited ways of searching or pulling subsets of data out for review. Once these problems start to appear, it's a good idea to transfer the data to a database created by a database management system (DBMS), such as Access.

SIMPLIFIED TEXT:
A database helps keep track of things like who they are, what they make, how many they sell, and other details. It stores all this information in one place where everyone can find it easily. Sometimes, though, lists get messy because some parts repeat themselves or don't match up right. This makes it hard to read everything at once

In [47]:
# Record quality review results for P011-P015

review_results = {
    "P011": (
        "Yes",
        "Yes",
        "Preserves the purpose of databases, problems with growing lists, "
        "and the role of a database management system."
    ),
    "P012": (
        "Yes",
        "Yes",
        "Explains database tables, reducing duplicated data, "
        "and normalization."
    ),
    "P013": (
        "No",
        "No",
        "Misrepresents the example of opening an order form and entering "
        "a customer's order. The final example is incorrect."
    ),
    "P014": (
        "Yes",
        "No",
        "Preserves retrieving and filtering data, but incorrectly suggests "
        "the warning about changes applies to filters generally rather "
        "than specifically to updateable queries."
    ),
    "P015": (
        "No",
        "No",
        "Misstates how macros and modules are created and confuses the role "
        "of VBA. The toy analogies are misleading."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
print("Passages reviewed in this group:", len(review_results))

display(
    review_df[
        review_df["Passage_ID"].isin(review_results.keys())
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

Review results saved!
Passages reviewed in this group: 5


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
10,P011,Yes,Yes,"Preserves the purpose of databases, problems w..."
11,P012,Yes,Yes,"Explains database tables, reducing duplicated ..."
12,P013,No,No,Misrepresents the example of opening an order ...
13,P014,Yes,No,"Preserves retrieving and filtering data, but i..."
14,P015,No,No,Misstates how macros and modules are created a...


## 19. Review Passages P016–P020

Passages P016–P020 will be reviewed by comparing the original technical text with the simplified version. Each passage will be evaluated for meaning preservation and factual accuracy.

The review will identify important omissions, changes in technical details, and unsupported information. The findings will be saved in the existing quality review worksheet.

In [48]:
for _, row in review_df[
    review_df["Passage_ID"].isin(
        ["P016", "P017", "P018", "P019", "P020"]
    )
].iterrows():

    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])
    print()

Passage ID: P016
Category: Networking

ORIGINAL TEXT:
When you consider moving to the cloud and taking on a role as an Azure developer, solution architect, or administrator, understanding networking fundamentals becomes crucial. It's essential that you grasp the foundations of how networking works and how to use different devices to build networks. While networks are made up of various network devices, there are other aspects to keep in mind as well. For example, network-communication protocols require devices to have network addresses assigned. You also need to secure your network to protect your organization's assets and resources.

SIMPLIFIED TEXT:
When thinking about working with Azure as a developer, architect, or admin, it’s super important to know all about networking basics. This means learning how networks work and how to use different gadgets to make them. Networks are like big playgrounds where lots of things can connect together. But just having toys isn’t enough – you also

In [49]:
# Record quality review results for P016-P020

review_results = {
    "P016": (
        "No",
        "No",
        "Omits the need for network addresses and changes network "
        "security into a vague playground-safety analogy."
    ),
    "P017": (
        "No",
        "No",
        "Oversimplifies LAN ownership and WAN connections. It does not "
        "clearly explain that WANs connect multiple LANs across locations."
    ),
    "P018": (
        "Yes",
        "Yes",
        "Clearly explains the purpose of network protocols, "
        "communication rules, and error handling."
    ),
    "P019": (
        "No",
        "No",
        "Omits the open-standard aspect and oversimplifies TCP/IP routing "
        "and connection behavior. Claiming it always knows the best path "
        "is misleading."
    ),
    "P020": (
        "No",
        "No",
        "Misrepresents DNS caching as remembering website appearance "
        "and oversimplifies authoritative DNS servers and name resolution."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
print("Passages reviewed in this group:", len(review_results))

display(
    review_df[
        review_df["Passage_ID"].isin(review_results.keys())
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

Review results saved!
Passages reviewed in this group: 5


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
15,P016,No,No,Omits the need for network addresses and chang...
16,P017,No,No,Oversimplifies LAN ownership and WAN connectio...
17,P018,Yes,Yes,Clearly explains the purpose of network protoc...
18,P019,No,No,Omits the open-standard aspect and oversimplif...
19,P020,No,No,Misrepresents DNS caching as remembering websi...


## 20. Review Passages P021–P025

Passages P021–P025 will be reviewed by comparing the original technical text with the simplified version. Each passage will be evaluated for meaning preservation and factual accuracy.

The review will identify important omissions, changes in technical details, and unsupported information. The findings will be saved in the existing quality review worksheet.

In [50]:
for _, row in review_df[
    review_df["Passage_ID"].isin(
        ["P021", "P022", "P023", "P024", "P025"]
    )
].iterrows():

    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])
    print()

Passage ID: P021
Category: Security

ORIGINAL TEXT:
Authentication is a security process that verifies a user's identity before granting access to apps, services, devices, or networks.While traditional MFA with SMS, email OTP, or authenticator apps significantly improves security over password-only systems, these options introduce friction — requiring additional steps for users, like entering codes, approving push notifications, or using authenticator apps. Moreover, these MFA options are prone to remote phishing attacks. In a remote phishing attack, attackers use social engineering and AI-driven tools to steal identity credentials — like passwords or one-time codes — without physical access to a user's device.

SIMPLIFIED TEXT:
Authentication is how we make sure someone is who they say they are before letting them into special places on computers or phones. It's better than just using a secret word (like your name) because it asks people to do extra things, like send a code or look at

In [51]:
# Record quality review results for P021-P025

review_results = {
    "P021": (
        "No",
        "No",
        "Omits specific MFA methods and remote phishing details. "
        "Incorrectly suggests a person's name is an example of a secret password."
    ),
    "P022": (
        "No",
        "No",
        "The simplified output is in Chinese rather than English, "
        "so it does not meet the intended output language."
    ),
    "P023": (
        "Yes",
        "No",
        "Preserves the general inheritance hierarchy but adds unsupported "
        "explanations and substitutes ECC for the original ECDiffieHellman example."
    ),
    "P024": (
        "Yes",
        "No",
        "Preserves most key encryption concepts, but the explanation of "
        "encrypting protocols such as TCP or UDP is potentially misleading."
    ),
    "P025": (
        "Yes",
        "Yes",
        "Preserves the main purpose of a firewall and its traffic-filtering "
        "role. The guard analogy is simplified but generally appropriate."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
print("Passages reviewed in this group:", len(review_results))

display(
    review_df[
        review_df["Passage_ID"].isin(review_results.keys())
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

Review results saved!
Passages reviewed in this group: 5


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
20,P021,No,No,Omits specific MFA methods and remote phishing...
21,P022,No,No,The simplified output is in Chinese rather tha...
22,P023,Yes,No,Preserves the general inheritance hierarchy bu...
23,P024,Yes,No,"Preserves most key encryption concepts, but th..."
24,P025,Yes,Yes,Preserves the main purpose of a firewall and i...


## 21. Review Passages P026–P030

Passages P026–P030 will be reviewed by comparing the original technical text with the simplified version. Each passage will be evaluated for meaning preservation and factual accuracy.

The review will identify important omissions, changes in technical details, and unsupported information. The findings will be saved in the existing quality review worksheet.

In [52]:
for _, row in review_df[
    review_df["Passage_ID"].isin(
        ["P026", "P027", "P028", "P029", "P030"]
    )
].iterrows():

    print("=" * 80)
    print("Passage ID:", row["Passage_ID"])
    print("Category:", row["Category"])

    print("\nORIGINAL TEXT:")
    print(row["Original_Text"])

    print("\nSIMPLIFIED TEXT:")
    print(row["Simplified_Text"])
    print()

Passage ID: P026
Category: Software Engineering

ORIGINAL TEXT:
Architecture strategies for standardizing tools and processes. This guide describes the recommendations for defining standards for software development tools and processes. Defining consistent practices leads to an efficient workload team and high-quality work. High-performing teams use industry-proven tools and processes to minimize wasted effort and potential code errors. The first step of optimizing development practices is standardizing tools and processes. When possible, use industry-proven solutions rather than developing in-house solutions. To further optimize your practices, adopt low-code and no-code tools. These tools enable you to focus efforts on your application and help you save time. For all tools and processes that you standardize, implement training so your teams understand and use them efficiently. To define standards that help optimize your development practices, consider the following recommendations.



In [53]:
review_results = {
    "P026": (
        "Yes",
        "No",
        "Preserves the main recommendation to standardize tools and train teams, but omits the specific role of low-code and no-code tools and incorrectly suggests they require little training."
    ),
    "P027": (
        "No",
        "No",
        "Omits important tool functions, including CI/CD pipelines, repositories, and specific testing types. Several descriptions are too vague to preserve the original technical meaning."
    ),
    "P028": (
        "No",
        "No",
        "Explains trunk-based development and branch protection generally but omits much of the AI-related guidance. The description of keeping all changes in one big place oversimplifies trunk-based development."
    ),
    "P029": (
        "No",
        "No",
        "Preserves the general purpose of a style guide but omits important standards for frameworks, naming conventions, environment differences, and evaluating new technologies."
    ),
    "P030": (
        "Yes",
        "No",
        "Preserves version visibility and the value of established design patterns, but introduces a confusing camera analogy and omits version exposure through REST interfaces and schema metadata."
    )
}

for passage_id, (meaning, accuracy, notes) in review_results.items():
    mask = review_df["Passage_ID"] == passage_id
    review_df.loc[mask, "Meaning_Preserved"] = meaning
    review_df.loc[mask, "Factually_Accurate"] = accuracy
    review_df.loc[mask, "Review_Notes"] = notes

review_df.to_csv(review_path, index=False)

print("Review results saved!")
print("Passages reviewed in this group:", len(review_results))

display(
    review_df[
        review_df["Passage_ID"].isin(review_results.keys())
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

Review results saved!
Passages reviewed in this group: 5


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
25,P026,Yes,No,Preserves the main recommendation to standardi...
26,P027,No,No,"Omits important tool functions, including CI/C..."
27,P028,No,No,Explains trunk-based development and branch pr...
28,P029,No,No,Preserves the general purpose of a style guide...
29,P030,Yes,No,Preserves version visibility and the value of ...


## 22. Calculate Overall Quality Review Results

The quality review results for all 30 passages will be summarized to measure how often the simplified text preserves the original meaning and remains factually accurate.

The summary will include counts and percentages for meaning preservation, factual accuracy, and passages that meet both criteria. These results will help explain the strengths and limitations of the language model alongside the readability measurements.

The summary will be saved as a CSV file for use in the final project report.

In [54]:
import pandas as pd

# Standardize review labels
review_df["Meaning_Preserved"] = (
    review_df["Meaning_Preserved"].astype(str).str.strip()
)
review_df["Factually_Accurate"] = (
    review_df["Factually_Accurate"].astype(str).str.strip()
)

total = len(review_df)

meaning_yes = (review_df["Meaning_Preserved"] == "Yes").sum()
accuracy_yes = (review_df["Factually_Accurate"] == "Yes").sum()

both_yes = (
    (review_df["Meaning_Preserved"] == "Yes") &
    (review_df["Factually_Accurate"] == "Yes")
).sum()

both_no = (
    (review_df["Meaning_Preserved"] == "No") &
    (review_df["Factually_Accurate"] == "No")
).sum()

summary_df = pd.DataFrame({
    "Measure": [
        "Total passages reviewed",
        "Meaning preserved",
        "Factually accurate",
        "Both meaning preserved and factually accurate",
        "Neither criterion met"
    ],
    "Count": [
        total,
        meaning_yes,
        accuracy_yes,
        both_yes,
        both_no
    ]
})

summary_df["Percentage"] = (
    summary_df["Count"] / total * 100
).round(1)

summary_path = (
    "/content/drive/MyDrive/COMP365_Team_Project/"
    "Quality_Review_Summary.csv"
)

summary_df.to_csv(summary_path, index=False)

print("Overall quality review summary:")
display(summary_df)

print("\nSummary saved to:")
print(summary_path)

Overall quality review summary:


,Measure,Count,Percentage
0,Total passages reviewed,30,100.0
1,Meaning preserved,16,53.3
2,Factually accurate,9,30.0
3,Both meaning preserved and factually accurate,9,30.0
4,Neither criterion met,14,46.7



Summary saved to:
/content/drive/MyDrive/COMP365_Team_Project/Quality_Review_Summary.csv


## 23. Save Readability Results

The readability measurements for all 30 passages will be saved in a separate CSV file. The file will preserve the original and simplified readability scores so they can be examined alongside the quality review results.

The readability results and quality review results provide complementary evidence about the model's performance. Readability scores measure how easy text is to read, while the quality review evaluates whether technical meaning and factual information are preserved.

In [55]:
# Save readability measurements to Google Drive
readability_path = (
    "/content/drive/MyDrive/COMP365_Team_Project/"
    "Readability_Results.csv"
)

readability_df.to_csv(readability_path, index=False)

print("Readability results saved!")
print("Number of passages:", len(readability_df))
print("\nColumns included:")
print(readability_df.columns.tolist())
print("\nSaved to:")
print(readability_path)

display(readability_df.head())

Readability results saved!
Number of passages: 30

Columns included:
['Passage_ID', 'Category', 'Source_URL', 'Original_Text', 'Word_Count', 'Simplified_Text', 'Original_Flesch_Reading_Ease', 'Simplified_Flesch_Reading_Ease', 'Original_Grade_Level', 'Simplified_Grade_Level', 'Reading_Ease_Change', 'Grade_Level_Change']

Saved to:
/content/drive/MyDrive/COMP365_Team_Project/Readability_Results.csv


,Passage_ID,Category,Source_URL,Original_Text,Word_Count,Simplified_Text,Original_Flesch_Reading_Ease,Simplified_Flesch_Reading_Ease,Original_Grade_Level,Simplified_Grade_Level,Reading_Ease_Change,Grade_Level_Change
0,P001,Cloud Computing,https://learn.microsoft.com/en-us/azure/cloud-...,Microsoft Azure is Microsoft's cloud platform ...,46,Microsoft Azure is like a big playground where...,14.290000,68.311421,16.980000,8.782623,54.021421,-8.197377
1,P002,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,When you send a request through any of the Azu...,53,"When you use any Azure tool, like an API or SD...",46.839182,75.976932,11.115094,6.077955,29.137749,-5.037140
2,P003,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,All capabilities that are available in the Azu...,38,All things you can do in the Azure portal work...,36.160526,78.460526,12.935789,7.035789,42.300000,-5.900000
3,P004,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource - A manageable item that's available ...,33,A resource is like a tool you can use in Azure...,24.164583,26.945000,12.538750,18.444444,2.780417,5.905694
4,P005,Cloud Computing,https://learn.microsoft.com/en-us/azure/azure-...,resource group - A container that holds relate...,46,A resource group is like a special box where y...,61.890000,80.785000,8.353333,6.960000,18.895000,-1.393333


## 24. Verify Project Files

The project folder will be checked to confirm that the dataset, simplified passages, quality review worksheet, quality review summary, and readability results have been saved.

This final check helps ensure that the main project outputs are available for the final report and submission.

In [56]:
import os

project_folder = "/content/drive/MyDrive/COMP365_Team_Project"

print("Project folder:", project_folder)
print("\nFiles currently saved:\n")

files = sorted(os.listdir(project_folder))

for filename in files:
    full_path = os.path.join(project_folder, filename)

    if os.path.isfile(full_path):
        size_kb = os.path.getsize(full_path) / 1024
        print(f"{filename} ({size_kb:.1f} KB)")
    else:
        print(f"{filename}/")

print("\nRequired output file check:")

required_files = [
    "Passages.csv",
    "Simplified_Passages.csv",
    "Quality_Review.csv",
    "Quality_Review_Summary.csv",
    "Readability_Results.csv"
]

for filename in required_files:
    status = "FOUND" if os.path.isfile(
        os.path.join(project_folder, filename)
    ) else "MISSING"

    print(f"{filename}: {status}")

Project folder: /content/drive/MyDrive/COMP365_Team_Project

Files currently saved:

COMP365_Technical_Language_Simplifier_Prototype.ipynb (162.8 KB)
Passages.csv (22.6 KB)
Quality_Review.csv (41.7 KB)
Quality_Review_Summary.csv (0.2 KB)
Readability_Results.csv (43.4 KB)
Simplified_Passages.csv (40.1 KB)

Required output file check:
Passages.csv: FOUND
Simplified_Passages.csv: FOUND
Quality_Review.csv: FOUND
Quality_Review_Summary.csv: FOUND
Readability_Results.csv: FOUND


## 25. Final Project Check

The final project check verifies that the 30 technical passages have been simplified and reviewed, the readability results have been saved, and the required project files are available.

The results will be used to summarize the project's achievements and limitations. The project is considered complete when the main outputs are saved and the notebook can be reviewed.

In [57]:
import os
import pandas as pd

project_folder = "/content/drive/MyDrive/COMP365_Team_Project"

required_files = [
    "Passages.csv",
    "Simplified_Passages.csv",
    "Quality_Review.csv",
    "Quality_Review_Summary.csv",
    "Readability_Results.csv",
    "COMP365_Technical_Language_Simplifier_Prototype.ipynb"
]

print("FINAL PROJECT CHECK")
print("=" * 40)

for filename in required_files:
    exists = os.path.isfile(os.path.join(project_folder, filename))
    print(f"{'FOUND' if exists else 'MISSING'}: {filename}")

review_df = pd.read_csv(
    os.path.join(project_folder, "Quality_Review.csv")
)
readability_df = pd.read_csv(
    os.path.join(project_folder, "Readability_Results.csv")
)

print("\nPassages reviewed:", len(review_df))
print("Readability records:", len(readability_df))

print(
    "Completed reviews:",
    (
        review_df["Meaning_Preserved"].isin(["Yes", "No"]) &
        review_df["Factually_Accurate"].isin(["Yes", "No"])
    ).sum()
)

print("\nProject check finished.")

FINAL PROJECT CHECK
FOUND: Passages.csv
FOUND: Simplified_Passages.csv
FOUND: Quality_Review.csv
FOUND: Quality_Review_Summary.csv
FOUND: Readability_Results.csv
FOUND: COMP365_Technical_Language_Simplifier_Prototype.ipynb

Passages reviewed: 30
Readability records: 30
Completed reviews: 30

Project check finished.


Jordyn Walsh-Hill — Human Quality Evaluation & Audit
Purpose: Assess whether the AI-generated simplifications preserve technical meaning and factual accuracy, identify failure patterns, and compare results with the team's evaluation targets.

In [58]:

import pandas as pd

# Load the completed human quality review
review_df = pd.read_csv(
    "/content/drive/MyDrive/COMP365_Team_Project/Quality_Review.csv"
).fillna("")

total = len(review_df)

meaning_yes = (review_df["Meaning_Preserved"] == "Yes").sum()
accuracy_yes = (review_df["Factually_Accurate"] == "Yes").sum()

both_yes = (
    (review_df["Meaning_Preserved"] == "Yes") &
    (review_df["Factually_Accurate"] == "Yes")
).sum()

print("JORDYN WALSH-HILL: HUMAN QUALITY EVALUATION")
print("=" * 55)
print("Total passages reviewed:", total)
print(f"Meaning preserved: {meaning_yes}/{total} ({meaning_yes/total*100:.1f}%)")
print(f"Factually accurate: {accuracy_yes}/{total} ({accuracy_yes/total*100:.1f}%)")
print(f"Both criteria satisfied: {both_yes}/{total} ({both_yes/total*100:.1f}%)")

print("\nREVIEW EVIDENCE: SELECTED PASSAGES")
print("=" * 55)

display(
    review_df[
        review_df["Passage_ID"].isin(["P001", "P008", "P022"])
    ][
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

print("\nREVIEW COMPLETE:", len(review_df) == 30)


JORDYN WALSH-HILL: HUMAN QUALITY EVALUATION
Total passages reviewed: 30
Meaning preserved: 16/30 (53.3%)
Factually accurate: 9/30 (30.0%)
Both criteria satisfied: 9/30 (30.0%)

REVIEW EVIDENCE: SELECTED PASSAGES


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
0,P001,No,No,"Adds unsupported claims about games, children,..."
7,P008,No,No,Omits that an ARM template is a JSON file and ...
21,P022,No,No,The simplified output is in Chinese rather tha...



REVIEW COMPLETE: True



## Jordyn Walsh-Hill — Checkpoint 3 Human Quality Evaluation

This section evaluates whether the simplified passages preserve the original technical meaning and remain factually accurate.

The review uses two criteria:

- **Meaning Preserved:** The simplified passage retains the essential technical information, important details, and original meaning.
- **Factually Accurate:** The simplified passage does not introduce unsupported claims, incorrect information, or changes to technical facts.

The purpose of this evaluation is to identify weaknesses in the AI simplification process and determine what improvements are needed before the final presentation.


In [59]:

# Jordyn Walsh-Hill
# Checkpoint 3: Human Quality Evaluation

import pandas as pd

#Standardize review responses
review_df["Meaning_Preserved"] = (
    review_df["Meaning_Preserved"].astype(str).str.strip().str.lower()
)

review_df["Factually_Accurate"] = (
    review_df["Factually_Accurate"].astype(str).str.strip().str.lower()
)

# Count completed human evaluations
total = len(review_df)

meaning_yes = (
    review_df["Meaning_Preserved"] == "yes"
).sum()

accuracy_yes = (
    review_df["Factually_Accurate"] == "yes"
).sum()

both_yes = (
    (review_df["Meaning_Preserved"] == "yes") &
    (review_df["Factually_Accurate"] == "yes")
).sum()

print("JORDYN WALSH-HILL: HUMAN QUALITY EVALUATION")
print("=" * 50)

print("Total passages reviewed:", total)

print(
    f"Meaning preserved: {meaning_yes}/{total} "
    f"({meaning_yes/total*100:.1f}%)"
)

print(
    f"Factually accurate: {accuracy_yes}/{total} "
    f"({accuracy_yes/total*100:.1f}%)"
)

print(
    f"Both criteria satisfied: {both_yes}/{total} "
    f"({both_yes/total*100:.1f}%)"
)


JORDYN WALSH-HILL: HUMAN QUALITY EVALUATION
Total passages reviewed: 30
Meaning preserved: 16/30 (53.3%)
Factually accurate: 9/30 (30.0%)
Both criteria satisfied: 9/30 (30.0%)


In [60]:

#Identify passages that failed one or both quality criteria
failed_passages = review_df[
    (review_df["Meaning_Preserved"] != "yes") |
    (review_df["Factually_Accurate"] != "yes")
].copy()

print("PASSAGES NEEDING QUALITY IMPROVEMENT")
print("=" * 50)
display(
    failed_passages[
        [
            "Passage_ID",
            "Meaning_Preserved",
            "Factually_Accurate",
            "Review_Notes"
        ]
    ]
)

print("\nTotal passages needing improvement:", len(failed_passages))


PASSAGES NEEDING QUALITY IMPROVEMENT


,Passage_ID,Meaning_Preserved,Factually_Accurate,Review_Notes
0,P001,no,no,"Adds unsupported claims about games, children,..."
3,P004,yes,no,"Most examples are preserved, but describing a ..."
7,P008,no,no,Omits that an ARM template is a JSON file and ...
8,P009,yes,no,Explains the general purpose of a Bicep file b...
9,P010,no,no,Does not clearly explain that an extension res...
12,P013,no,no,Misrepresents the example of opening an order ...
13,P014,yes,no,"Preserves retrieving and filtering data, but i..."
14,P015,no,no,Misstates how macros and modules are created a...
15,P016,no,no,Omits the need for network addresses and chang...
16,P017,no,no,Oversimplifies LAN ownership and WAN connectio...



Total passages needing improvement: 21


In [61]:

#Display selected examples from the human review

selected_ids = ["P001", "P008", "P022"]
for passage_id in selected_ids:

    passage = review_df[
        review_df["Passage_ID"] == passage_id
    ]
    if not passage.empty:
        row = passage.iloc[0]

        print("\n" + "=" * 70)
        print("PASSAGE:", passage_id)
        print("MEANING PRESERVED:", row["Meaning_Preserved"])
        print("FACTUALLY ACCURATE:", row["Factually_Accurate"])

        print("\nORIGINAL TEXT:")
        print(row["Original_Text"])

        print("\nSIMPLIFIED TEXT:")
        print(row["Simplified_Text"])

        print("\nREVIEW NOTES:")
        print(row["Review_Notes"])



PASSAGE: P001
MEANING PRESERVED: no
FACTUALLY ACCURATE: no

ORIGINAL TEXT:
Microsoft Azure is Microsoft's cloud platform designed to simplify the process of building, deploying, and managing applications and services through a global network of Microsoft-managed data centers. It offers a wide range of services that allow businesses and developers to create scalable, reliable, and maintainable applications.

SIMPLIFIED TEXT:
Microsoft Azure is like a big playground where they put all their computer stuff together in special places called data centers. This helps people make fun things on computers without having to worry too much about how it works. They have lots of different games (services) you can play with, which lets grown-ups build cool apps and kids learn new skills.

REVIEW NOTES:
Adds unsupported claims about games, children, and learning skills. Omits building, deploying, and managing applications.

PASSAGE: P008
MEANING PRESERVED: no
FACTUALLY ACCURATE: no

ORIGINAL TEXT:
A

In [71]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import pandas as pd

review_df = pd.read_csv("Quality_Review.csv")

print("Review file loaded successfully!")
print("Total rows:", len(review_df))
display(review_df.head())

### AI Disclosure

Generative AI was used in this project to simplify 30 technical passages using the Qwen2.5-1.5B-Instruct language model. AI assistance was also used to support coding, readability analysis, and the organization of project results. The simplified passages were manually reviewed for meaning preservation and factual accuracy. The evaluation identified cases where technical details were omitted, meaning was changed, or unsupported information was introduced. AI-generated outputs were not assumed to be fully accurate, and the final results were reviewed before being reported.